In [9]:
# There are 2 types of Agents - 
"""
1. Simple Agent:
    The agent enters a loop where it sends a request and checks whether the response contains a tool call.
    If so, it executes the function (e.g. get_weather) and appends the result to the conversation.
    The loop stops when there are no more tool calls and the response contains final text (output_text).
2. Objective-Based Agent:
    The agent is given a custom objective function (e.g. check whether the phrase "task complete" is in the output).
    It loops until the objective function returns True.
"""
import os, sys
sys.path.append(r"C:\Users\allan\projects\Prompt_Engineering")
from llm_config import MODEL_GROQ, groq_api_key 
from getpass import getpass
import requests
import json

In [15]:
def get_weather(latitude, longtitude):
    response = request.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    )
    data = response.json()
    return data['current']['temperature_2m']

weather_tool = {
    "type" : "function",
    "name" : "get_weather",
    "description" : "Get the current weather of the city",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "latitude" : {"type" : "number", "description" : "Latitude of the place"},
            "longitude" : {"type" : "number", "description" : "longitude of the place"}           
        },
        "required" : ["latitude", "longtitude"],
        "additionalparameters" : False,
    },
    "strict" : True
}
tools = [weather_tool]

In [18]:
def get_capital(country):
    response = request.get(
        f"https://api.restcountries.com/countries/v5/names.common?{country}"
    )
    data = response.json()
    return data['capital'] 

capital_tool = {
    "type" : "function ",
    "name" : "get_capital",
    "description" : "Get the capital of the country",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "capital" : {"type" : "number", "description" : "capital of the country"}         
        },
        "required" : ["capital"],
        "additionalparameters" : False,
    },
    "strict" : True
}
tools = [capital_tool]

In [19]:
# This agent sends a prompt (asking about the weather), then enters an agentic loop.
# At each turn, it calls the Responses API:

# 1. If the response contains a tool call: The agent executes the function (using our get_weather tool) and 
# appends the function result to the conversation as a new message.
# 2. If the response provides output text: The agent stops, printing the final output.
"""
             ┌─────────────────────┐
             │ User asks question  │
             │ California weather? │
             └──────────┬──────────┘
                        ↓
                 ┌─────────────┐
                 │     LLM     │
                 └──────┬──────┘
                        ↓
              "I need weather tool"
                        ↓
                 ┌─────────────┐
                 │ Python runs │
                 │ get_weather │
                 └──────┬──────┘
                        ↓
                    15°C
                        ↓
              Give 15°C to the LLM
                        ↓
                 ┌─────────────┐
                 │     LLM     │
                 └──────┬──────┘
                        ↓
          "California is currently 15°C."
                        ↓
                       STOP

"""
from llm_config import groq


messages = [{"role" : "user", "content" : "Whats a weather in capital of france today?"}]
while True:
    response = groq.responses.create(
        model = "openai/gpt-oss-120b",
        input = messages,
        tools = tools
    )

    if response.output:
        for output_item in response.output:
            if hasattr(output_item, 'type') and output_item.type == "function_call":
                messages.append(output_item)
                tool_call = output_item
                args = json.loads(tool_call.arguments)

                result = get_country(args['country'])
                print(f"Executed {tool_call.name}: Result Country capital {result}")
                messages.append({
                    "type" : "function_call_output",
                    "call_id" : tool_call.call_id,
                    "output" :str(result)
                }),
                result = get_weather(args['latitude'],args['longitude'])
                print(f"Executed {tool_call.name}: Result {result} cels")
                messages.append({
                    "type" : "function_call_output",
                    "call_id" : tool_call.call_id,
                    "output" :str(result)
                })   

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}